# Clustering Preserving QAT

## Install TensorFlow Model Optimization Toolkit
* 설치 완료 후 반드시 Runtime 재시작!
    * '런타임' > '세션 다시 시작' 메뉴 선택

In [ ]:
!pip install tensorflow-model-optimization

## Mount Google driver

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


In [ ]:
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
else :
  save_dir = '../files/save/'

## Import Module

In [ ]:
import tensorflow as tf
import numpy as np

import tensorflow_model_optimization as tfmot
from tensorflow_model_optimization.python.core.keras.compat import keras

import tempfile

print(tf.__version__)
print(np.__version__)

2.19.0
1.26.4


## Load Dataset

In [ ]:
(train_images, train_labels), (test_images, test_labels) = keras.datasets.mnist.load_data()

train_images = (train_images / 255.0).astype(np.float32)
test_images = (test_images / 255.0).astype(np.float32)

11490434/11490434 [==============================] - 0s 0us/step


In [ ]:
train_images = np.expand_dims(train_images, axis=-1)
test_images = np.expand_dims(test_images, axis=-1)

## Load Baseline Model for MNIST

In [ ]:
model = keras.models.load_model(save_dir + 'baseline_model.h5')
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 26, 26, 32)        320       
                                                                 
 max_pooling2d (MaxPooling2  (None, 13, 13, 32)        0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 11, 11, 16)        4624      
                                                                 
 max_pooling2d_1 (MaxPoolin  (None, 5, 5, 16)          0         
 g2D)                                                            
                                                                 
 flatten (Flatten)           (None, 400)               0         
                                                                 
 dense (Dense)               (None, 128)               5

In [ ]:
_, baseline_model_accuracy = model.evaluate(
    test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)

Baseline test accuracy: 0.9904000163078308


## Cluster and fine-tune
* number of cluster : 8
* cluster centroids init : K-means++

In [ ]:
clustering_params = {
  'number_of_clusters': 8,
  'cluster_centroids_init': tfmot.clustering.keras.CentroidInitialization.KMEANS_PLUS_PLUS,
  'cluster_per_channel' : True
}

clustered_model = tfmot.clustering.keras.cluster_weights(model, **clustering_params)

# Use smaller learning rate for fine-tuning
clustered_model.compile(
  loss=keras.losses.SparseCategoricalCrossentropy(),
  optimizer=keras.optimizers.Adam(learning_rate=1e-5),
  metrics=['accuracy'])

clustered_model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 cluster_conv2d (ClusterWei  (None, 26, 26, 32)        864       
 ghts)                                                           
                                                                 
 cluster_max_pooling2d (Clu  (None, 13, 13, 32)        0         
 sterWeights)                                                    
                                                                 
 cluster_conv2d_1 (ClusterW  (None, 11, 11, 16)        9360      
 eights)                                                         
                                                                 
 cluster_max_pooling2d_1 (C  (None, 5, 5, 16)          0         
 lusterWeights)                                                  
                                                                 
 cluster_flatten (ClusterWe  (None, 400)               0

## Fine tune the model for clustering

In [ ]:
clustered_model.fit(
  train_images,
  train_labels,
  epochs=3,
  validation_split=0.1)

Epoch 1/3
1688/1688 [==============================] - 25s 10ms/step - loss: 0.0059 - accuracy: 0.9979 - val_loss: 0.0403 - val_accuracy: 0.9917
Epoch 2/3
1688/1688 [==============================] - 16s 9ms/step - loss: 0.0039 - accuracy: 0.9988 - val_loss: 0.0390 - val_accuracy: 0.9922
Epoch 3/3
1688/1688 [==============================] - 16s 9ms/step - loss: 0.0032 - accuracy: 0.9992 - val_loss: 0.0390 - val_accuracy: 0.9925


## Cluster 확인

In [ ]:
def print_model_weight_clusters(model):
    for layer in model.layers:
        if isinstance(layer, keras.layers.Wrapper):
            weights = layer.trainable_weights
        else:
            weights = layer.weights
        for weight in weights:
            # ignore auxiliary quantization weights
            if "quantize_layer" in weight.name:
                continue
            if "kernel" in weight.name:
                unique_count = len(np.unique(weight))
                print(
                    f"{layer.name}/{weight.name}: {unique_count} clusters "
                )

In [ ]:
stripped_clustered_model = tfmot.clustering.keras.strip_clustering(clustered_model)

print_model_weight_clusters(stripped_clustered_model)

conv2d/kernel:0: 256 clusters 
conv2d_1/kernel:0: 128 clusters 
dense/kernel:0: 8 clusters 
dense_1/kernel:0: 8 clusters 


In [ ]:
_, clustered_model_accuracy = clustered_model.evaluate(
  test_images, test_labels, verbose=0)

print('Baseline test accuracy:', baseline_model_accuracy)
print('Clustered test accuracy:', clustered_model_accuracy)

Baseline test accuracy: 0.9904000163078308
Clustered test accuracy: 0.9918000102043152


## QAT VS CQAT
* QAT : training 과정에서 cluster 파괴
* CQAT : training 과정에서도 cluster 유지

In [ ]:
# QAT
qat_model = tfmot.quantization.keras.quantize_model(stripped_clustered_model)

qat_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

qat_model.summary()

print('Train QAT model:')
qat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer (QuantizeLa  (None, 28, 28, 1)         3         
 yer)                                                            
                                                                 
 quant_conv2d (QuantizeWrap  (None, 26, 26, 32)        387       
 perV2)                                                          
                                                                 
 quant_max_pooling2d (Quant  (None, 13, 13, 32)        1         
 izeWrapperV2)                                                   
                                                                 
 quant_conv2d_1 (QuantizeWr  (None, 11, 11, 16)        4659      
 apperV2)                                                        
                                                                 
 quant_max_pooling2d_1 (Qua  (None, 5, 5, 16)          1

In [ ]:
# CQAT
quant_aware_annotate_model = tfmot.quantization.keras.quantize_annotate_model(
              stripped_clustered_model)
cqat_model = tfmot.quantization.keras.quantize_apply(
              quant_aware_annotate_model,
              tfmot.experimental.combine.Default8BitClusterPreserveQuantizeScheme())

cqat_model.compile(optimizer='adam',
              loss=keras.losses.SparseCategoricalCrossentropy(),
              metrics=['accuracy'])

cqat_model.summary()

print('Train CQAT Model:')
cqat_model.fit(train_images, train_labels, batch_size=128, epochs=1, validation_split=0.1)

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 quantize_layer_1 (Quantize  (None, 28, 28, 1)         3         
 Layer)                                                          
                                                                 
 quant_conv2d (QuantizeWrap  (None, 26, 26, 32)        1219      
 perV2)                                                          
                                                                 
 quant_max_pooling2d (Quant  (None, 13, 13, 32)        1         
 izeWrapperV2)                                                   
                                                                 
 quant_conv2d_1 (QuantizeWr  (None, 11, 11, 16)        14003     
 apperV2)                                                        
                                                                 
 quant_max_pooling2d_1 (Qua  (None, 5, 5, 16)          1

422/422 [==============================] - 16s 16ms/step - loss: 0.0087 - accuracy: 0.9984 - val_loss: 0.0397 - val_accuracy: 0.9903


## Cluster수 확인

In [ ]:
print("CQAT Model clusters:")
print_model_weight_clusters(cqat_model)
print()
print("QAT Model clusters:")
print_model_weight_clusters(qat_model)

CQAT Model clusters:
quant_conv2d/conv2d/kernel:0: 256 clusters 
quant_conv2d_1/conv2d_1/kernel:0: 128 clusters 
quant_dense/dense/kernel:0: 8 clusters 
quant_dense_1/dense_1/kernel:0: 8 clusters 

QAT Model clusters:
quant_conv2d/conv2d/kernel:0: 288 clusters 
quant_conv2d_1/conv2d_1/kernel:0: 4325 clusters 
quant_dense/dense/kernel:0: 43742 clusters 
quant_dense_1/dense_1/kernel:0: 1176 clusters 


## LiteRT 모델로 변환 (QAT)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
tflite_qat_file = save_dir + 'mnist_cqat_x.tflite'
open(tflite_qat_file, 'wb').write(tflite_model)

63888

## LiteRT 모델로 변환 (CQAT)

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(cqat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/convert.py:854: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
tflite_cqat_file = save_dir + 'mnist_cqat.tflite'
open(tflite_cqat_file, 'wb').write(tflite_model)

63888

## 모델 압축 테스트

* 압축 함수 정의

In [ ]:
import zipfile
import os

def get_gzipped_model_size(file):
  # It returns the size of the gzipped model in kilobytes.

  _, zipped_file = tempfile.mkstemp('.zip')
  with zipfile.ZipFile(zipped_file, 'w', compression=zipfile.ZIP_DEFLATED) as f:
    f.write(file)

  return os.path.getsize(zipped_file)/1000

* 압축된 파일 크기 비교

In [ ]:
print("QAT model size: ", get_gzipped_model_size(tflite_qat_file), ' KB')
print("CQAT model size: ", get_gzipped_model_size(tflite_cqat_file), ' KB')

QAT model size:  52.833  KB
CQAT model size:  29.384  KB


## LiteRT 설치 및 Interpreter 로딩

In [ ]:
!pip install ai-edge-litert

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 59.3 MB/s eta 0:00:00


In [ ]:
from ai_edge_litert.interpreter import Interpreter

## interpreter 생성 (CQAT)

In [ ]:
interpreter_cqat = Interpreter(model_path=str(tflite_cqat_file))
interpreter_cqat.allocate_tensors()

## input/output dtype 확인 (CQAT)

In [ ]:
input_dtype = interpreter_cqat.get_input_details()[0]['dtype']
output_dtype = interpreter_cqat.get_output_details()[0]['dtype']

print("input dtype : {}".format(input_dtype))
print("output dtype : {}".format(output_dtype))

input dtype : <class 'numpy.float32'>
output dtype : <class 'numpy.float32'>


## Test data 기반 accuracy 평가

In [ ]:
def eval_model(interpreter):
  input_details = interpreter.get_input_details()[0]
  output_details = interpreter.get_output_details()[0]
  input_index = input_details["index"]
  output_index = output_details["index"]

  prediction_digits = []
  for i, test_image in enumerate(test_images):
    test_image = np.expand_dims(test_image, axis=0).astype(input_details['dtype'])
    interpreter.set_tensor(input_index, test_image)

    interpreter.invoke()

    output = interpreter.get_tensor(output_index)
    digit = np.argmax(output)
    prediction_digits.append(digit)

  prediction_digits = np.array(prediction_digits)
  accuracy = (prediction_digits == test_labels).mean()
  return accuracy

In [ ]:
cqat_test_accuracy = eval_model(interpreter_cqat)

print('Clustered and quantized TFLite test_accuracy :', cqat_test_accuracy)
print('Baseline model test_accuracy :', baseline_model_accuracy)

Clustered and quantized TFLite test_accuracy : 0.9916
Baseline model test_accuracy : 0.9904000163078308
